In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# heavy packages
from transformers import pipeline

In [ ]:
## lighter packages
import torch
import pandas as pd
import os

In [37]:
file_path = r"/content/drive/MyDrive/WVU_academic/EE-593A/project/input_files/MA_monthlytweets.csv"
dir_name = os.path.dirname(file_path)
base_name = os.path.basename(file_path)
file_root, ext = os.path.splitext(base_name)
output_name = f"{file_root}_with_outage{ext}"
output_path = os.path.join(dir_name, output_name)
df   = pd.read_csv(file_path)
df["Text"] = df["Text"].str.replace('"', "'", regex=False) ## replacing double quote "" with single ''
df.head()

,Screen Name,Status ID,Text,Local Timestamp,Likes,Retweets,State
0,BOS311,2.050441e+16,"Closed report at 169-199 Mountfort St, Brookli...",10:40 AM - 30 Dec 2010,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,MA
1,BOS311,2.049531e+16,"Closed report at 157-167 Milk St, Boston: http...",10:03 AM - 30 Dec 2010,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,MA
2,pyyhkala,1.944947e+16,Out about Acton appears not even like a 5 year...,12:48 PM - 27 Dec 2010,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,MA
3,thepennyhoarder,1.082664e+18,There's a secret bank paying 100x the normal i...,10:41 AM - 8 Jan 2019,Like\n \n \n 14\n \n \n \n ...,Retweet\n \n \n 2\n \n \n \n ...,MA
4,Skeetz8,1.940686e+16,Yoo this storm is #killing niggas. Ain't shit ...,9:58 AM - 27 Dec 2010,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n 3\n \n \n \n ...,MA


In [ ]:
device = 0 if torch.cuda.is_available() else -1

zeroshot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

# quick check
print("Using device:", "cuda" if device>=0 else "cpu")

In [38]:
def get_outage_status(text):
    result = zeroshot(text, candidate_labels=["power outage", "not outage"])
    top_label = result["labels"][0]
    return 0 if top_label == "power outage" else 1

In [ ]:
df["Outage_status"] = df["Text"].apply(get_outage_status)

In [ ]:
df.to_csv(output_path, index=False)

In [16]:
# tweet = "New Year's resolution: smoke less...get blackout drunk more often"
tweet = "As a Yanks-Jets fan tonight is a perfect storm. Jets v. Ravens on MNF and CC Sabathia (Yankees) v. David Price (Rays). I love sports."
result = zeroshot(
    tweet,
    candidate_labels=["power outage", "not outage"]
)
print(result)

{'sequence': 'As a Yanks-Jets fan tonight is a perfect storm. Jets v. Ravens on MNF and CC Sabathia (Yankees) v. David Price (Rays). I love sports.', 'labels': ['not outage', 'power outage'], 'scores': [0.8414157032966614, 0.158584326505661]}


In [17]:
print(result["labels"][0], result["scores"][0])

not outage 0.8414157032966614
